# Hugging Face Transformers 微调语言模型-问答任务

我们已经学会使用 Pipeline 加载支持问答任务的预训练模型，本教程代码将展示如何微调训练一个支持问答任务的模型。

**注意：微调后的模型仍然是通过提取上下文的子串来回答问题的，而不是生成新的文本。**

### 模型执行问答效果示例

![Widget inference representing the QA task](docs/images/question_answering.png)

In [44]:
# 根据你使用的模型和GPU资源情况，调整以下关键参数
squad_v2 = False
model_checkpoint = "distilbert-base-uncased"
batch_size = 16

## 下载数据集

在本教程中，我们将使用[斯坦福问答数据集(SQuAD）](https://rajpurkar.github.io/SQuAD-explorer/)。

### SQuAD 数据集

**斯坦福问答数据集(SQuAD)** 是一个阅读理解数据集，由众包工作者在一系列维基百科文章上提出问题组成。每个问题的答案都是相应阅读段落中的文本片段或范围，或者该问题可能无法回答。

SQuAD2.0将SQuAD1.1中的10万个问题与由众包工作者对抗性地撰写的5万多个无法回答的问题相结合，使其看起来与可回答的问题类似。要在SQuAD2.0上表现良好，系统不仅必须在可能时回答问题，还必须确定段落中没有支持任何答案，并放弃回答。

In [2]:
from datasets import load_dataset

In [3]:
datasets = load_dataset("squad_v2" if squad_v2 else "squad")


The `datasets` object itself is [`DatasetDict`](https://huggingface.co/docs/datasets/package_reference/main_classes.html#datasetdict), which contains one key for the training, validation and test set.

数据集对象本身是一个 `DatasetDict`，其中包含分别对应训练集、验证集和测试集的键。

In [4]:
datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

#### 对比数据集

相比快速入门使用的 Yelp 评论数据集，我们可以看到 SQuAD 训练和测试集都新增了用于上下文、问题以及问题答案的列：

**YelpReviewFull Dataset：**

```json

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})
```

In [5]:
datasets["train"][10]

{'id': '5733bed24776f41900661188',
 'title': 'University_of_Notre_Dame',
 'context': 'The university is the major seat of the Congregation of Holy Cross (albeit not its official headquarters, which are in Rome). Its main seminary, Moreau Seminary, is located on the campus across St. Joseph lake from the Main Building. Old College, the oldest building on campus and located near the shore of St. Mary lake, houses undergraduate seminarians. Retired priests and brothers reside in Fatima House (a former retreat center), Holy Cross House, as well as Columba Hall near the Grotto. The university through the Moreau Seminary has ties to theologian Frederick Buechner. While not Catholic, Buechner has praised writers from Notre Dame and Moreau Seminary created a Buechner Prize for Preaching.',
 'question': 'Where is the headquarters of the Congregation of the Holy Cross?',
 'answers': {'text': ['Rome'], 'answer_start': [119]}}

#### 从上下文中组织回复内容

我们可以看到答案是通过它们在文本中的起始位置（这里是第515个字符）以及它们的完整文本表示的，这是上面提到的上下文的子字符串。

In [6]:
from datasets import ClassLabel, Sequence
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)
    
    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
        elif isinstance(typ, Sequence) and isinstance(typ.feature, ClassLabel):
            df[column] = df[column].transform(lambda x: [typ.feature.names[i] for i in x])
    display(HTML(df.to_html()))

In [7]:
show_random_elements(datasets["train"])

,id,title,context,question,answers
0,57112cf3a58dae1900cd6cfe,Nintendo_Entertainment_System,"The NES Test Station's front features a Game Pak slot and connectors for testing various components (AC adapter, RF switch, Audio/Video cable, NES Control Deck, accessories and games), with a centrally-located selector knob to chose which component to test. The unit itself weighs approximately 11.7 pounds without a TV. It connects to a television via a combined A/V and RF Switch cable. By actuating the green button, a user can toggle between an A/V Cable or RF Switch connection. The television it is connected to (typically 11"" to 14"") is meant to be placed atop it.",Where was the Television meant to be placed?,"{'text': ['atop'], 'answer_start': [563]}"
1,56f89fc39b226e1400dd0ce9,Gene,"Two chains of DNA twist around each other to form a DNA double helix with the phosphate-sugar backbone spiralling around the outside, and the bases pointing inwards with adenine base pairing to thymine and guanine to cytosine. The specificity of base pairing occurs because adenine and thymine align form two hydrogen bonds, whereas cytosine and guanine form three hydrogen bonds. The two strands in a double helix must therefore be complementary, with their sequence of bases matching such that the adenines of one strand are paired with the thymines of the other strand, and so on.:4.1",What is the backbone of a DNA double helix made of?,"{'text': ['phosphate-sugar'], 'answer_start': [78]}"
2,5733a9144776f41900660f8e,Anthropology,"Sporadic use of the term for some of the subject matter occurred subsequently, such as the use by Étienne Serres in 1838 to describe the natural history, or paleontology, of man, based on comparative anatomy, and the creation of a chair in anthropology and ethnography in 1850 at the National Museum of Natural History (France) by Jean Louis Armand de Quatrefages de Bréau. Various short-lived organizations of anthropologists had already been formed. The Société Ethnologique de Paris, the first to use Ethnology, was formed in 1839. Its members were primarily anti-slavery activists. When slavery was abolished in France in 1848 the Société was abandoned.",When was a chair created for anthropology at the National Museum of Natural History?,"{'text': ['1850'], 'answer_start': [272]}"
3,57268c25dd62a815002e8928,Greece,"The Greek national basketball team has a decades-long tradition of excellence in the sport, being considered among the world's top basketball powers. As of 2012, it ranked 4th in the world and 2nd in Europe. They have won the European Championship twice in 1987 and 2005, and have reached the final four in two of the last four FIBA World Championships, taking the second place in the world in 2006 FIBA World Championship, after a spectacular 101–95 win against Team USA in the tournament's semifinal. The domestic top basketball league, A1 Ethniki, is composed of fourteen teams. The most successful Greek teams are Olympiacos, Panathinaikos, Aris Thessaloniki, AEK Athens and P.A.O.K. Greek basketball teams are the most successful in European basketball the last 25 years, having won as many as 9 Euroleagues since the establishment of the modern era Euroleague Final Four format in 1988, while no other nation has won more than 4 Euroleague championships in this period. Besides the 9 Euroleagues, Greek basketball teams (Panathinaikos, Olympiacos, Aris Thessaloniki, AEK Athens, P.A.O.K, Maroussi) have won 3 Triple Crowns, 5 Saporta Cups, 2 Korać Cups and 1 FIBA Europe Champions Cup. After the 2005 European Championship triumph of the Greek national basketball team, Greece became the reigning European Champion in both football and basketball.",How many times has the Greek national basketball team won the European championship?,"{'text': ['twice'], 'answer_start': [248]}"
4,5726723f5951b619008f72e3,Mexico_City,"In 1980 half of all the industrial jobs in Mexico were located in Mexico City. Under relentless growth, the Mexico City govern

## 预处理数据

In [8]:
from transformers import AutoTokenizer
    
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


以下断言确保我们的 Tokenizers 使用的是 FastTokenizer（Rust 实现，速度和功能性上有一定优势）。

In [9]:
import transformers
assert isinstance(tokenizer, transformers.PreTrainedTokenizerFast)

您可以在大模型表上查看哪种类型的模型具有可用的快速标记器，哪种类型没有。

您可以直接在两个句子上调用此标记器（一个用于答案，一个用于上下文）：

In [10]:
tokenizer("What is your name?", "My name is Sylvain.")

{'input_ids': [101, 2054, 2003, 2115, 2171, 1029, 102, 2026, 2171, 2003, 25353, 22144, 2378, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

### Tokenizer 进阶操作

在问答预处理中的一个特定问题是如何处理非常长的文档。

在其他任务中，当文档的长度超过模型最大句子长度时，我们通常会截断它们，但在这里，删除上下文的一部分可能会导致我们丢失正在寻找的答案。

为了解决这个问题，我们允许数据集中的一个（长）示例生成多个输入特征，每个特征的长度都小于模型的最大长度（或我们设置的超参数）。

In [11]:
# The maximum length of a feature (question and context)
max_length = 384 
# The authorized overlap between two part of the context when splitting it is needed.
doc_stride = 128 

#### 超出最大长度的文本数据处理

下面，我们从训练集中找出一个超过最大长度（384）的文本：

In [12]:
for i, example in enumerate(datasets["train"]):
    if len(tokenizer(example["question"], example["context"])["input_ids"]) > 384:
        break
# 挑选出来超过384（最大长度）的数据样例
example = datasets["train"][i]

In [13]:
len(tokenizer(example["question"], example["context"])["input_ids"])

396

#### 截断上下文不保留超出部分

In [14]:
len(tokenizer(example["question"],
              example["context"],
              max_length=max_length,
              truncation="only_second")["input_ids"])

384

#### 关于截断的策略

- 直接截断超出部分: truncation=`only_second`
- 仅截断上下文（context），保留问题（question）：`return_overflowing_tokens=True` & 设置`stride`


In [15]:
tokenized_example = tokenizer(
    example["question"],
    example["context"],
    max_length=max_length,
    truncation="only_second",
    return_overflowing_tokens=True,
    stride=doc_stride
)

使用此策略截断后，Tokenizer 将返回多个 `input_ids` 列表。

In [16]:
[len(x) for x in tokenized_example["input_ids"]]

[384, 157]

解码两个输入特征，可以看到重叠的部分：

In [17]:
for x in tokenized_example["input_ids"][:2]:
    print(tokenizer.decode(x))

[CLS] how many wins does the notre dame men's basketball team have? [SEP] the men's basketball team has over 1, 600 wins, one of only 12 schools who have reached that mark, and have appeared in 28 ncaa tournaments. former player austin carr holds the record for most points scored in a single game of the tournament with 61. although the team has never won the ncaa tournament, they were named by the helms athletic foundation as national champions twice. the team has orchestrated a number of upsets of number one ranked teams, the most notable of which was ending ucla's record 88 - game winning streak in 1974. the team has beaten an additional eight number - one teams, and those nine wins rank second, to ucla's 10, all - time in wins against the top team. the team plays in newly renovated purcell pavilion ( within the edmund p. joyce center ), which reopened for the beginning of the 2009 – 2010 season. the team is coached by mike brey, who, as of the 2014 – 15 season, his fifteenth at notr

#### 使用 offsets_mapping 获取原始的 input_ids

设置 `return_offsets_mapping=True`，将使得截断分割生成的多个 input_ids 列表中的 token，通过映射保留原始文本的 input_ids。

如下所示：第一个标记（[CLS]）的起始和结束字符都是（0, 0），因为它不对应问题/答案的任何部分，然后第二个标记与问题(question)的字符0到3相同.

In [18]:
tokenized_example = tokenizer(
    example["question"],
    example["context"],
    max_length=max_length,
    truncation="only_second",
    return_overflowing_tokens=True,
    # 标记出同一个问题
    return_offsets_mapping=True,
    stride=doc_stride
)
print(tokenized_example["offset_mapping"][0][:100])

[(0, 0), (0, 3), (4, 8), (9, 13), (14, 18), (19, 22), (23, 28), (29, 33), (34, 37), (37, 38), (38, 39), (40, 50), (51, 55), (56, 60), (60, 61), (0, 0), (0, 3), (4, 7), (7, 8), (8, 9), (10, 20), (21, 25), (26, 29), (30, 34), (35, 36), (36, 37), (37, 40), (41, 45), (45, 46), (47, 50), (51, 53), (54, 58), (59, 61), (62, 69), (70, 73), (74, 78), (79, 86), (87, 91), (92, 96), (96, 97), (98, 101), (102, 106), (107, 115), (116, 118), (119, 121), (122, 126), (127, 138), (138, 139), (140, 146), (147, 153), (154, 160), (161, 165), (166, 171), (172, 175), (176, 182), (183, 186), (187, 191), (192, 198), (199, 205), (206, 208), (209, 210), (211, 217), (218, 222), (223, 225), (226, 229), (230, 240), (241, 245), (246, 248), (248, 249), (250, 258), (259, 262), (263, 267), (268, 271), (272, 277), (278, 281), (282, 285), (286, 290), (291, 301), (301, 302), (303, 307), (308, 312), (313, 318), (319, 321), (322, 325), (326, 330), (330, 331), (332, 340), (341, 351), (352, 354), (355, 363), (364, 373), (374,

因此，我们可以使用这个映射来找到答案在给定特征中的起始和结束标记的位置。

我们只需区分偏移的哪些部分对应于问题，哪些部分对应于上下文。

In [19]:
first_token_id = tokenized_example["input_ids"][0][1]
offsets = tokenized_example["offset_mapping"][0][1]
print(tokenizer.convert_ids_to_tokens([first_token_id])[0], example["question"][offsets[0]:offsets[1]])

how How


In [20]:
second_token_id = tokenized_example["input_ids"][0][2]
offsets = tokenized_example["offset_mapping"][0][2]
print(tokenizer.convert_ids_to_tokens([second_token_id])[0], example["question"][offsets[0]:offsets[1]])

many many


In [21]:
example["question"]

"How many wins does the Notre Dame men's basketball team have?"

借助`tokenized_example`的`sequence_ids`方法，我们可以方便的区分token的来源编号：

- 对于特殊标记：返回None，
- 对于正文Token：返回句子编号（从0开始编号）。

综上，现在我们可以很方便的在一个输入特征中找到答案的起始和结束 Token。

In [22]:
sequence_ids = tokenized_example.sequence_ids()
print(sequence_ids)

[None, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, None, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [23]:
answers = example["answers"]
start_char = answers["answer_start"][0]
end_char = start_char + len(answers["text"][0])

# 当前span在文本中的起始标记索引。
token_start_index = 0
while sequence_ids[token_start_index] != 1:
    token_start_index += 1

# 当前span在文本中的结束标记索引。
token_end_index = len(tokenized_example["input_ids"][0]) - 1
while sequence_ids[token_end_index] != 1:
    token_end_index -= 1

# 检测答案是否超出span范围（如果超出范围，该特征将以CLS标记索引标记）。
offsets = tokenized_example["offset_mapping"][0]
if (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
    # 将token_start_index和token_end_index移动到答案的两端。
    # 注意：如果答案是最后一个单词，我们可以移到最后一个标记之后（边界情况）。
    while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
        token_start_index += 1
    start_position = token_start_index - 1
    while offsets[token_end_index][1] >= end_char:
        token_end_index -= 1
    end_position = token_end_index + 1
    print(start_position, end_position)
else:
    print("答案不在此特征中。")


23 26


打印检查是否准确找到了起始位置：

In [24]:
# 通过查找 offset mapping 位置，解码 context 中的答案 
print(tokenizer.decode(tokenized_example["input_ids"][0][start_position: end_position+1]))
# 直接打印 数据集中的标准答案（answer["text"])
print(answers["text"][0])

over 1, 600
over 1,600


#### 关于填充的策略

- 对于没有超过最大长度的文本，填充补齐长度。
- 对于需要左侧填充的模型，交换 question 和 context 顺序

In [25]:
pad_on_right = tokenizer.padding_side == "right"

### 整合以上所有预处理步骤

让我们将所有内容整合到一个函数中，并将其应用到训练集。

针对不可回答的情况（上下文过长，答案在另一个特征中），我们为开始和结束位置都设置了cls索引。

如果allow_impossible_answers标志为False，我们还可以简单地从训练集中丢弃这些示例。

In [26]:
def prepare_train_features(examples):
    # 一些问题的左侧可能有很多空白字符，这对我们没有用，而且会导致上下文的截断失败
    # （标记化的问题将占用大量空间）。因此，我们删除左侧的空白字符。
    examples["question"] = [q.lstrip() for q in examples["question"]]

    # 使用截断和填充对我们的示例进行标记化，但保留溢出部分，使用步幅（stride）。
    # 当上下文很长时，这会导致一个示例可能提供多个特征，其中每个特征的上下文都与前一个特征的上下文有一些重叠。
    tokenized_examples = tokenizer(
        examples["question" if pad_on_right else "context"],
        examples["context" if pad_on_right else "question"],
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # 由于一个示例可能给我们提供多个特征（如果它具有很长的上下文），我们需要一个从特征到其对应示例的映射。这个键就提供了这个映射关系。
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    # 偏移映射将为我们提供从令牌到原始上下文中的字符位置的映射。这将帮助我们计算开始位置和结束位置。
    offset_mapping = tokenized_examples.pop("offset_mapping")

    # 让我们为这些示例进行标记！
    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        # 我们将使用 CLS 特殊 token 的索引来标记不可能的答案。
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        # 获取与该示例对应的序列（以了解上下文和问题是什么）。
        sequence_ids = tokenized_examples.sequence_ids(i)

        # 一个示例可以提供多个跨度，这是包含此文本跨度的示例的索引。
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]
        # 如果没有给出答案，则将cls_index设置为答案。
        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            # 答案在文本中的开始和结束字符索引。
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            # 当前跨度在文本中的开始令牌索引。
            token_start_index = 0
            while sequence_ids[token_start_index] != (1 if pad_on_right else 0):
                token_start_index += 1

            # 当前跨度在文本中的结束令牌索引。
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != (1 if pad_on_right else 0):
                token_end_index -= 1

            # 检测答案是否超出跨度（在这种情况下，该特征的标签将使用CLS索引）。
            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
            else:
                # 否则，将token_start_index和token_end_index移到答案的两端。
                # 注意：如果答案是最后一个单词（边缘情况），我们可以在最后一个偏移之后继续。
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized_examples["start_positions"].append(token_start_index - 1)
                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples["end_positions"].append(token_end_index + 1)

    return tokenized_examples

#### datasets.map 的进阶使用

使用 `datasets.map` 方法将 `prepare_train_features` 应用于所有训练、验证和测试数据：

- batched: 批量处理数据。
- remove_columns: 因为预处理更改了样本的数量，所以在应用它时需要删除旧列。
- load_from_cache_file：是否使用datasets库的自动缓存

datasets 库针对大规模数据，实现了高效缓存机制，能够自动检测传递给 map 的函数是否已更改（因此需要不使用缓存数据）。如果在调用 map 时设置 `load_from_cache_file=False`，可以强制重新应用预处理。

In [27]:
tokenized_datasets = datasets.map(prepare_train_features,
                                  batched=True,
                                  remove_columns=datasets["train"].column_names)

## 微调模型

现在我们的数据已经准备好用于训练，我们可以下载预训练模型并进行微调。

由于我们的任务是问答，我们使用 `AutoModelForQuestionAnswering` 类。(对比 Yelp 评论打分使用的是 `AutoModelForSequenceClassification` 类）

警告通知我们正在丢弃一些权重（`vocab_transform` 和 `vocab_layer_norm` 层），并随机初始化其他一些权重（`pre_classifier` 和 `classifier` 层）。在微调模型情况下是绝对正常的，因为我们正在删除用于预训练模型的掩码语言建模任务的头部，并用一个新的头部替换它，对于这个新头部，我们没有预训练的权重，所以库会警告我们在用它进行推理之前应该对这个模型进行微调，而这正是我们要做的事情。

In [28]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### 训练超参数（TrainingArguments）

In [45]:
batch_size=64
model_dir = f"models/{model_checkpoint}-finetuned-squad"

args = TrainingArguments(
    output_dir=model_dir,
    evaluation_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
)

#### Data Collator（数据整理器）

数据整理器将训练数据整理为批次数据，用于模型训练时的批次处理。本教程使用默认的 `default_data_collator`。


In [30]:
from transformers import default_data_collator

data_collator = default_data_collator

### 实例化训练器（Trainer）

为了减少训练时间（需要大量算力支持），我们不在本教程的训练模型过程中计算模型评估指标。

而是训练完成后，再独立进行模型评估。

In [31]:
trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

#### GPU 使用情况

训练数据与模型配置：

- SQUAD v1.1
- model_checkpoint = "distilbert-base-uncased"
- batch_size = 64

NVIDIA GPU 使用情况：

```shell
Every 1.0s: nvidia-smi                                                   Wed Dec 20 15:39:57 2023

Wed Dec 20 15:39:57 2023
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:0D.0 Off |                    0 |
| N/A   67C    P0              67W /  70W |  14617MiB / 15360MiB |    100%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+

+---------------------------------------------------------------------------------------+
| Processes:                                                                            |
|  GPU   GI   CI        PID   Type   Process name                            GPU Memory |
|        ID   ID                                                             Usage      |
|=======================================================================================|
|    0   N/A  N/A     16384      C   /root/miniconda3/bin/python               14612MiB |
+---------------------------------------------------------------------------------------+
```

In [32]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.485400,1.264354
2,1.116400,1.161018
3,0.973200,1.159791


TrainOutput(global_step=4152, training_loss=1.3000807716207927, metrics={'train_runtime': 1744.4736, 'train_samples_per_second': 152.236, 'train_steps_per_second': 2.38, 'total_flos': 2.602335381127373e+16, 'train_loss': 1.3000807716207927, 'epoch': 3.0})

### 训练完成后，第一时间保存模型权重文件。

In [49]:
import os

# QA 子目录
qa_dir = os.path.join(model_dir, "QA")
os.makedirs(qa_dir, exist_ok=True)

# 保存模型、tokenizer、训练状态
trainer.save_model(qa_dir)
trainer.tokenizer.save_pretrained(qa_dir)   # 关键：tokenizer 也要保存到同一目录
trainer.save_state()

In [47]:
from pathlib import Path
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# 训练时模型保存根目录
qa_dir = os.path.join(model_dir, "QA")
os.makedirs(qa_dir, exist_ok=True)

# 要保存的根目录
model_dir_save = Path(f"/sigan/ai-ml/models/finetuned/{model_checkpoint}-finetuned-squad")
# QA 子目录
qa_dir_save = model_dir_save / "QA"
qa_dir_save.mkdir(parents=True, exist_ok=True)

# 从原目录加载模型和 tokenizer（注意 tokenizer 也从 model_dir 加载）
model = AutoModelForSequenceClassification.from_pretrained(qa_dir)
tokenizer = AutoTokenizer.from_pretrained(qa_dir)

# 保存到 QA 子目录
model.save_pretrained(qa_dir_save)
tokenizer.save_pretrained(qa_dir_save)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at models/distilbert-base-uncased-finetuned-squad/QA and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


('/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad/QA/tokenizer_config.json',
 '/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad/QA/special_tokens_map.json',
 '/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad/QA/vocab.txt',
 '/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad/QA/added_tokens.json',
 '/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad/QA/tokenizer.json')

## 模型评估

**评估模型输出需要一些额外的处理：将模型的预测映射回上下文的部分。**

模型直接输出的是预测答案的`起始位置`和`结束位置`的**logits**

In [51]:
import torch
from accelerate import Accelerator
from accelerate.state import AcceleratorState

# 重置并重新初始化 Accelerator 状态
AcceleratorState._reset_state()
Accelerator()

for batch in trainer.get_eval_dataloader():
    break
batch = {k: v.to(trainer.args.device) for k, v in batch.items()}
with torch.no_grad():
    output = trainer.model(**batch)
output.keys()

odict_keys(['loss', 'start_logits', 'end_logits'])

模型的输出是一个类似字典的对象，其中包含损失（因为我们提供了标签），以及起始和结束logits。我们不需要损失来进行预测，让我们看一下logits：

In [52]:
output.start_logits.shape, output.end_logits.shape

(torch.Size([64, 384]), torch.Size([64, 384]))

In [53]:
output.start_logits.argmax(dim=-1), output.end_logits.argmax(dim=-1)

(tensor([ 46,  57,  78,  43, 118, 108,  72,  35, 108,  34,  73,  41,  80,  91,
         156,  35,  83,  91,  80,  58,  77,  31,  42,  53,  41,  35,  42,  77,
          11,  44,  27, 133,  66,  40,  87,  44,  43,  41, 127,  26,  28,  33,
          87, 127,  95,  25,  43, 132,  42,  29,  44,  46,  24,  44,  65,  58,
          81,  14,  59,  72,  25,  36,  57,  43], device='cuda:0'),
 tensor([ 47,  58,  81,  44, 118, 110,  75,  37, 110,  36,  76,  42,  83,  94,
         158,  35,  83,  94,  83,  60,  80,  31,  43,  54,  42,  35,  43,  80,
          13,  45,  28, 133,  66,  41,  89,  45,  87,  42, 127,  27,  30,  34,
          89, 127,  97,  26,  44, 132,  43,  30,  45,  47,  25,  45,  65,  59,
          81,  14,  60,  72,  25,  36,  58,  43], device='cuda:0'))

#### 如何从模型输出的位置 logit 组合成答案

我们有每个特征和每个标记的logit。在每个特征中为每个标记预测答案最明显的方法是，将起始logits的最大索引作为起始位置，将结束logits的最大索引作为结束位置。

在许多情况下这种方式效果很好，但是如果此预测给出了不可能的结果该怎么办？比如：起始位置可能大于结束位置，或者指向问题中的文本片段而不是答案。在这种情况下，我们可能希望查看第二好的预测，看它是否给出了一个可能的答案，并选择它。

选择第二好的答案并不像选择最佳答案那么容易：
- 它是起始logits中第二佳索引与结束logits中最佳索引吗？
- 还是起始logits中最佳索引与结束logits中第二佳索引？
- 如果第二好的答案也不可能，那么对于第三好的答案，情况会更加棘手。

为了对答案进行分类，
1. 将使用通过添加起始和结束logits获得的分数
1. 设计一个名为`n_best_size`的超参数，限制不对所有可能的答案进行排序。
1. 我们将选择起始和结束logits中的最佳索引，并收集这些预测的所有答案。
1. 在检查每一个是否有效后，我们将按照其分数对它们进行排序，并保留最佳的答案。

以下是我们如何在批次中的第一个特征上执行此操作的示例：

In [54]:
n_best_size = 20

In [55]:
import numpy as np

start_logits = output.start_logits[0].cpu().numpy()
end_logits = output.end_logits[0].cpu().numpy()

# 获取最佳的起始和结束位置的索引：
start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()

valid_answers = []

# 遍历起始位置和结束位置的索引组合
for start_index in start_indexes:
    for end_index in end_indexes:
        if start_index <= end_index:  # 需要进一步测试以检查答案是否在上下文中
            valid_answers.append(
                {
                    "score": start_logits[start_index] + end_logits[end_index],
                    "text": ""  # 我们需要找到一种方法来获取与上下文中答案对应的原始子字符串
                }
            )



然后，我们可以根据它们的得分对`valid_answers`进行排序，并仅保留最佳答案。唯一剩下的问题是如何检查给定的跨度是否在上下文中（而不是问题中），以及如何获取其中的文本。为此，我们需要向我们的验证特征添加两个内容：

- 生成该特征的示例的ID（因为每个示例可以生成多个特征，如前所示）；
- 偏移映射，它将为我们提供从标记索引到上下文中字符位置的映射。

这就是为什么我们将使用以下函数稍微不同于`prepare_train_features`来重新处理验证集：

In [68]:
def prepare_validation_features(examples):
    # 一些问题的左侧有很多空白，这些空白并不有用且会导致上下文截断失败（分词后的问题会占用很多空间）。
    # 因此我们移除这些左侧空白
    # 空白本身没有语义，却会占用 token，可能把真正的上下文挤出去
    examples["question"] = [q.lstrip() for q in examples["question"]]

    # 使用截断和可能的填充对我们的示例进行分词，但使用步长保留溢出的令牌。这导致一个长上下文的示例可能产生
    # 几个特征，每个特征的上下文都会稍微与前一个特征的上下文重叠。
    tokenized_examples = tokenizer(
        examples["question" if pad_on_right else "context"],
        examples["context" if pad_on_right else "question"],
        truncation="only_second" if pad_on_right else "only_first", # 只截断第二段，也就是上下文；通常保留完整问题
        max_length=max_length, # 每个模型输入最多多少个 token
        stride=doc_stride, # 相邻片段保留一部分重叠内容，避免答案被切断
        return_overflowing_tokens=True, # 长上下文装不下时，拆成多个片段
        return_offsets_mapping=True, # 记录 token 对应原文中的字符位置
        padding="max_length", # 每个输入补齐到相同长度，方便组成 batch
    )

    # 由于一个示例在上下文很长时可能会产生几个特征，我们需要一个从特征映射到其对应示例的映射。这个键就是为了这个目的。
    # 记录“当前特征来自哪个原始样本”。模型不需要这个字段，所以通过 pop() 把它从输入数据中移除，但后续处理仍然要使用它。
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")

    # 我们保留产生这个特征的示例ID，并且会存储偏移映射。
    tokenized_examples["example_id"] = []

    for i in range(len(tokenized_examples["input_ids"])):
        # 获取与该示例对应的序列（以了解哪些是上下文，哪些是问题）。
        sequence_ids = tokenized_examples.sequence_ids(i) # 告诉程序：当前 token 属于问题、上下文，还是特殊符号
        context_index = 1 if pad_on_right else 0

        # 一个示例可以产生几个文本段，这里是包含该文本段的示例的索引。
        sample_index = sample_mapping[i]
        tokenized_examples["example_id"].append(examples["id"][sample_index])

        # 将不属于上下文的偏移映射设置为None，以便容易确定一个令牌位置是否属于上下文。
        # 这行把问题部分和 [CLS]、[SEP] 等特殊 token 的位置映射改成 None，只保留上下文部分的字符位置。
        # 原因是答案只能来自文章，不能来自问题。否则模型可能给问题里的某个词很高的起止分数，后处理却错误地把它当成答案。
        tokenized_examples["offset_mapping"][i] = [
            (o if sequence_ids[k] == context_index else None)
            for k, o in enumerate(tokenized_examples["offset_mapping"][i])
        ]

    return tokenized_examples


将`prepare_validation_features`应用到整个验证集：

In [57]:
validation_features = datasets["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=datasets["validation"].column_names
)

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

现在，我们可以使用 `Trainer.predict` 方法获取所有特征的预测结果：
Now we can grab the predictions for all features by using the `Trainer.predict` method:

- evaluate() 主要算 loss 和指标，不强调保留逐条原始预测。
- predict() 会返回原始预测，适合拿 logits 做自定义后处理。

In [69]:
# 执行纯预测或验证
raw_predictions = trainer.predict(validation_features)

`Trainer`会隐藏模型不使用的列（在这里是`example_id`和`offset_mapping`，我们需要它们进行后处理），所以我们需要将它们重新设置回来：

In [70]:
# 表格里的文件没有被删除，只是刚才被藏起来了；这行把 example_id 和 offset_mapping 两个抽屉重新打开
validation_features.set_format(type=validation_features.format["type"], columns=list(validation_features.features.keys()))

现在，我们可以改进之前的测试：

由于在偏移映射中，当它对应于问题的一部分时，我们将其设置为None，因此可以轻松检查答案是否完全在上下文中。我们还可以从考虑中排除非常长的答案（可以调整的超参数）。

展开说下具体实现：
- 首先从模型输出中获取起始和结束的逻辑值（logits），这些值表明答案在文本中可能开始和结束的位置。
- 然后，它使用偏移映射（offset_mapping）来找到这些逻辑值在原始文本中的具体位置。
- 接下来，代码遍历可能的开始和结束索引组合，排除那些不在上下文范围内或长度不合适的答案。
- 对于有效的答案，它计算出一个分数（基于开始和结束逻辑值的和），并将答案及其分数存储起来。
- 最后，它根据分数对答案进行排序，并返回得分最高的几个答案。

In [71]:
max_answer_length = 30

In [72]:
start_logits = output.start_logits[0].cpu().numpy()
end_logits = output.end_logits[0].cpu().numpy()
offset_mapping = validation_features[0]["offset_mapping"]

# 第一个特征来自第一个示例。对于更一般的情况，我们需要将example_id匹配到一个示例索引
context = datasets["validation"][0]["context"]

# 收集最佳开始/结束逻辑的索引：
start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()
valid_answers = []
for start_index in start_indexes:
    for end_index in end_indexes:
        # 不考虑超出范围的答案，原因是索引超出范围或对应于输入ID的部分不在上下文中。
        if (
            start_index >= len(offset_mapping)
            or end_index >= len(offset_mapping)
            or offset_mapping[start_index] is None
            or offset_mapping[end_index] is None
        ):
            continue
        # 不考虑长度小于0或大于max_answer_length的答案。
        if end_index < start_index or end_index - start_index + 1 > max_answer_length:
            continue
        if start_index <= end_index: # 我们需要细化这个测试，以检查答案是否在上下文中
            start_char = offset_mapping[start_index][0]
            end_char = offset_mapping[end_index][1]
            valid_answers.append(
                {
                    "score": start_logits[start_index] + end_logits[end_index],
                    "text": context[start_char: end_char]
                }
            )

valid_answers = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[:n_best_size]
valid_answers


[{'score': 15.211548, 'text': 'Denver Broncos'},
 {'score': 13.64353,
  'text': 'Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers'},
 {'score': 11.104723, 'text': 'Carolina Panthers'},
 {'score': 10.975087, 'text': 'Broncos'},
 {'score': 10.15848, 'text': 'Denver'},
 {'score': 9.40707,
  'text': 'Broncos defeated the National Football Conference (NFC) champion Carolina Panthers'},
 {'score': 8.862364,
  'text': 'American Football Conference (AFC) champion Denver Broncos'},
 {'score': 8.438874,
  'text': 'The American Football Conference (AFC) champion Denver Broncos'},
 {'score': 7.896212,
  'text': 'Denver Broncos defeated the National Football Conference'},
 {'score': 7.467263,
  'text': 'Denver Broncos defeated the National Football Conference (NFC)'},
 {'score': 7.294347,
  'text': 'American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers'},
 {'score': 7.0634556,
  

打印比较模型输出和标准答案（Ground-truth）是否一致:

In [73]:
datasets["validation"][0]["answers"]

{'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'],
 'answer_start': [177, 177, 177]}

**模型最高概率的输出与标准答案一致**

正如上面的代码所示，这在第一个特征上很容易，因为我们知道它来自第一个示例。

对于其他特征，我们需要建立一个示例与其对应特征的映射关系。

此外，由于一个示例可以生成多个特征，我们需要将由给定示例生成的所有特征中的所有答案汇集在一起，然后选择最佳答案。

下面的代码构建了一个示例索引到其对应特征索引的映射关系

**生活化例子：一篇文章被复印成了两张有重叠的纸。现在给每张纸贴上“原文章编号”，再把同一原文章的所有纸张放进一个文件夹**

比如：
    1. 原始样本 5 的 ID 是 "abc"
    2. 特征 20 和特征 21 都来自 "abc"
    3. 于是：features_per_example[5] == [20, 21]

In [74]:
import collections

# 原始验证集，一条对应一个问题、一个上下文和标准答案
examples = datasets["validation"]
# 分词后的模型输入。因为长上下文会被切成多个重叠片段，所以一个 example 可能产生多个 feature
features = validation_features

example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
features_per_example = collections.defaultdict(list)
for i, feature in enumerate(features):
    # 自动创建空列表的字典。遇到新编号时，它自动生成
    features_per_example[example_id_to_index[feature["example_id"]]].append(i)

当`squad_v2 = True`时，有一定概率出现不可能的答案（impossible answer)。

    1. SQuAD v1 的每个问题都能从文章中找到答案。
    2. SQuAD v2 增加了一种情况：文章中根本没有答案。

上面的代码仅保留在上下文中的答案，我们还需要获取不可能答案的分数（其起始和结束索引对应于CLS标记的索引）。

当一个示例生成多个特征时，我们必须在所有特征中的不可能答案都预测出现不可能答案时（因为一个特征可能之所以能够预测出不可能答案，是因为答案不在它可以访问的上下文部分），这就是为什么一个示例中不可能答案的分数是该示例生成的每个特征中的不可能答案的分数的最小值。

人话：一篇 30 页的合同被切成 3 段。只有当三段都没有找到答案时，才能说整份合同没有答案。任何一段找到了可用答案，都不能直接判定整份合同无答案。

In [75]:
from tqdm.auto import tqdm

# 1. 通过刚才的 features_per_example 找到它产生的所有特征。
# 2. 每个特征取开始分数最高的前 20 个位置。
# 3. 每个特征取结束分数最高的前 20 个位置。
# 4. 两两组合成最多 20 × 20 = 400 个候选边界。
# 5. 删除问题区域、特殊 token、结束位置在开始位置之前的候选。
# 6. 删除超过 30 个 token 的候选。
# 7. 通过 offset_mapping 把 token 位置换回原始上下文的字符区间。
# 8. 用下面的分数给答案排序：答案分数 = start_logit + end_logit
# 9. 对所有片段的所有候选答案统一比较，选出最高分答案。
# 10. SQuAD v2 中，再把最高答案分数和无答案分数比较。

# 最终返回一个字典
# {
#     "问题ID-1": "New York City",
#     "问题ID-2": "",
#     "问题ID-3": "Paris",
# }

# 本函数做的是
# logits
# → 候选开始/结束位置
# → 字符区间
# → 原文子串
# → 跨片段比较
# → 每个问题的最终答案
def postprocess_qa_predictions(examples, features, raw_predictions, n_best_size = 20, max_answer_length = 30):
    all_start_logits, all_end_logits = raw_predictions
    # 构建一个从示例到其对应特征的映射。
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature["example_id"]]].append(i)

    # 我们需要填充的字典。
    predictions = collections.OrderedDict()

    # 日志记录。
    print(f"正在后处理 {len(examples)} 个示例的预测，这些预测分散在 {len(features)} 个特征中。")

    # 遍历所有示例！
    for example_index, example in enumerate(tqdm(examples)):
        # 这些是与当前示例关联的特征的索引。
        feature_indices = features_per_example[example_index]

        min_null_score = None # 仅在squad_v2为True时使用。
        valid_answers = []
        
        context = example["context"]
        # 遍历与当前示例关联的所有特征。
        for feature_index in feature_indices:
            # 我们获取模型对这个特征的预测。
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            # 这将允许我们将logits中的某些位置映射到原始上下文中的文本跨度。
            offset_mapping = features[feature_index]["offset_mapping"]

            # 更新最小空预测。
            cls_index = features[feature_index]["input_ids"].index(tokenizer.cls_token_id)
            feature_null_score = start_logits[cls_index] + end_logits[cls_index]
            if min_null_score is None or min_null_score < feature_null_score:
                min_null_score = feature_null_score

            # 浏览所有的最佳开始和结束logits，为 `n_best_size` 个最佳选择。
            start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
            end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # 不考虑超出范围的答案，原因是索引超出范围或对应于输入ID的部分不在上下文中。
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                    ):
                        continue
                    # 不考虑长度小于0或大于max_answer_length的答案。
                    if end_index < start_index or end_index - start_index + 1 > max_answer_length:
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    valid_answers.append(
                        {
                            "score": start_logits[start_index] + end_logits[end_index],
                            "text": context[start_char: end_char]
                        }
                    )
        
        if len(valid_answers) > 0:
            best_answer = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[0]
        else:
            # 在极少数情况下我们没有一个非空预测，我们创建一个假预测以避免失败。
            best_answer = {"text": "", "score": 0.0}
        
        # 选择我们的最终答案：最佳答案或空答案（仅适用于squad_v2）
        if not squad_v2:
            predictions[example["id"]] = best_answer["text"]
        else:
            answer = best_answer["text"] if best_answer["score"] > min_null_score else ""
            predictions[example["id"]] = answer

    return predictions


在原始结果上应用后处理问答结果：

In [76]:
# datasets["validation"]	原始问题、上下文和答案
# validation_features	切好的模型输入与位置映射
# raw_predictions.predictions	每个特征的开始、结束位置分数
final_predictions = postprocess_qa_predictions(datasets["validation"], validation_features, raw_predictions.predictions)

正在后处理 10570 个示例的预测，这些预测分散在 10784 个特征中。


  0%|          | 0/10570 [00:00<?, ?it/s]

使用 `datasets.load_metric` 中加载 `SQuAD v2` 的评估指标

In [77]:
from datasets import load_metric

metric = load_metric("squad_v2" if squad_v2 else "squad")

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/datasets/load.py:752: FutureWarning: The repository for squad contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.16.1/metrics/squad/squad.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


接下来，我们可以调用上面定义的函数进行评估。

只需稍微调整一下预测和标签的格式，因为它期望的是一系列字典而不是一个大字典。

在使用`squad_v2`数据集时，我们还需要设置`no_answer_probability`参数（我们在这里将其设置为0.0，因为如果我们选择了答案，我们已经将答案设置为空）。
- `no_answer_probability`：表示模型认为“这个问题没有答案”的概率
- 写成 0.0 原因是 postprocess_qa_predictions() 已经根据无答案分数做了硬性判断：
    - 有答案：写入答案文本
    - 无答案：已经写入空字符串。
    - metric 只要看到 prediction_text == ""，就能判断这次预测的答案为空。
    - 但 0.0 只是一个占位值。SQuAD v2 还会利用这个概率计算 best_exact、best_f1 和最佳阈值；如果希望这些阈值指标有意义，应该传入真实的无答案概率，而不是统一写 0。

### 参数含义
1. Exact Match 缩写为 EM，表示预测答案与标准答案完全一致的比例
   - EM 更严格，检验答案边界和内容是否完全正确
3. F1 衡量预测答案和标准答案之间的 token 重合程度，即使没有完全一致，也能得到部分分数
   - F1 更宽松，观察答案是否包含足够正确的信息
   - EM 与 F1 差距大，常见原因是答案多带了或少带了一两个词
   - 两者都低，通常说明模型定位答案片段的能力较弱。

In [78]:
if squad_v2:
    formatted_predictions = [{"id": k, "prediction_text": v, "no_answer_probability": 0.0} for k, v in final_predictions.items()]
else:
    # final_predictions 是大字典，而 SQuAD metric 需要列表，所以这里把：{"abc": "Paris"} 改成 [{"id": "abc", "prediction_text": "Paris"}]
    formatted_predictions = [{"id": k, "prediction_text": v} for k, v in final_predictions.items()]
# 标准答案也整理成同样的结构
references = [{"id": ex["id"], "answers": ex["answers"]} for ex in datasets["validation"]]
# 然后计算指标
metric.compute(predictions=formatted_predictions, references=references)

{'exact_match': 74.52223273415326, 'f1': 83.53321694990905}

### Homework：加载本地保存的模型，进行评估和再训练更高的 F1 Score

In [50]:
trained_model = AutoModelForQuestionAnswering.from_pretrained(model_dir)

In [51]:
trained_trainer = Trainer(
    trained_model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


# SQuAD v2 训练

## 参数与数据集

v2 中有些样本的答案是空列表，这类样本代表“上下文中没有答案”
{
    "answers": {
        "answer_start": [],
        "text": []
    }
}

In [1]:
# 启用 SQuAD v2
squad_v2 = True
model_checkpoint = "distilbert-base-uncased"
batch_size = 16

max_length = 384
doc_stride = 128

In [2]:
from datasets import load_dataset

datasets = load_dataset("squad_v2")

print(datasets)
print(datasets["validation"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})
{'id': '56ddde6b9a695914005b9628', 'title': 'Normans', 'context': 'The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries gave their name to Normandy, a region in France. They were descended from Norse ("Norman" comes from "Norseman") raiders and pirates from Denmark, Iceland and Norway who, under their leader Rollo, agreed to swear fealty to King Charles III of West Francia. Through generations of assimilation and mixing with the native Frankish and Roman-Gaulish populations, their descendants would gradually merge with the Carolingian-based cultures of West Francia. The distinct cultural and ethnic identity of the Normans emerged initially in the first half of t

In [3]:
from transformers import AutoTokenizer

# 加载 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
pad_on_right = tokenizer.padding_side == "right"

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## 训练数据预处理

这个函数同时处理两种标签：
- 有答案：标记答案的起止 token。
- 无答案：开始、结束位置都标记为 [CLS]。

In [4]:
def prepare_train_features(examples):
    examples["question"] = [
        question.lstrip()
        for question in examples["question"]
    ]

    tokenized_examples = tokenizer(
        examples["question" if pad_on_right else "context"],
        examples["context" if pad_on_right else "question"],
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop(
        "overflow_to_sample_mapping"
    )
    offset_mapping = tokenized_examples.pop(
        "offset_mapping"
    )

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized_examples.sequence_ids(i)
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        # SQuAD v2 的无答案样本
        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != (
            1 if pad_on_right else 0
        ):
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != (
            1 if pad_on_right else 0
        ):
            token_end_index -= 1

        # 答案不在当前片段中：当前特征也视为无答案
        if not (
            offsets[token_start_index][0] <= start_char
            and offsets[token_end_index][1] >= end_char
        ):
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
            continue

        while (
            token_start_index < len(offsets)
            and offsets[token_start_index][0] <= start_char
        ):
            token_start_index += 1

        while offsets[token_end_index][1] >= end_char:
            token_end_index -= 1

        tokenized_examples["start_positions"].append(
            token_start_index - 1
        )
        tokenized_examples["end_positions"].append(
            token_end_index + 1
        )

    return tokenized_examples

In [5]:
# 应用预处理
tokenized_datasets = datasets.map(
    prepare_train_features,
    batched=True,
    remove_columns=datasets["train"].column_names,
    load_from_cache_file=False,
)

print(tokenized_datasets)

Map:   0%|          | 0/130319 [00:00<?, ? examples/s]

Map:   0%|          | 0/11873 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 131754
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 12134
    })
})


## 创建模型与训练参数

In [6]:
from transformers import (
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
)

model = AutoModelForQuestionAnswering.from_pretrained(
    model_checkpoint
)

training_output_dir = (
    "/sigan/ai-ml/models/finetuned/"
    f"{model_checkpoint}-finetuned-squad-v2"
)

args = TrainingArguments(
    output_dir=training_output_dir,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=3,
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)

/home/sigan/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.177200,1.341441
2,0.980300,1.281112
3,0.779600,1.468179


TrainOutput(global_step=24705, training_loss=1.1103514940771007, metrics={'train_runtime': 2888.9694, 'train_samples_per_second': 136.818, 'train_steps_per_second': 8.551, 'total_flos': 3.873165421863629e+16, 'train_loss': 1.1103514940771007, 'epoch': 3.0})

In [8]:
trainer.save_model(training_output_dir)
tokenizer.save_pretrained(training_output_dir)

('/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad-v2/tokenizer_config.json',
 '/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad-v2/special_tokens_map.json',
 '/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad-v2/vocab.txt',
 '/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad-v2/added_tokens.json',
 '/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad-v2/tokenizer.json')

## 验证集特征

In [9]:
def prepare_validation_features(examples):
    examples["question"] = [
        question.lstrip()
        for question in examples["question"]
    ]

    tokenized_examples = tokenizer(
        examples["question" if pad_on_right else "context"],
        examples["context" if pad_on_right else "question"],
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop(
        "overflow_to_sample_mapping"
    )
    tokenized_examples["example_id"] = []

    for i in range(
        len(tokenized_examples["input_ids"])
    ):
        sequence_ids = tokenized_examples.sequence_ids(i)
        context_index = 1 if pad_on_right else 0
        sample_index = sample_mapping[i]

        tokenized_examples["example_id"].append(
            examples["id"][sample_index]
        )

        tokenized_examples["offset_mapping"][i] = [
            offset if sequence_ids[k] == context_index else None
            for k, offset in enumerate(
                tokenized_examples["offset_mapping"][i]
            )
        ]

    return tokenized_examples

In [10]:
validation_features = datasets["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=datasets["validation"].column_names,
    load_from_cache_file=False,
)

Map:   0%|          | 0/11873 [00:00<?, ? examples/s]

## SQuAD v2 后处理

In [11]:
import collections
import numpy as np
from tqdm.auto import tqdm


def postprocess_qa_predictions(
    examples,
    features,
    raw_predictions,
    n_best_size=20,
    max_answer_length=30,
):
    all_start_logits, all_end_logits = raw_predictions

    example_id_to_index = {
        example_id: index
        for index, example_id in enumerate(examples["id"])
    }

    features_per_example = collections.defaultdict(list)

    for feature_index, feature in enumerate(features):
        original_index = example_id_to_index[
            feature["example_id"]
        ]
        features_per_example[original_index].append(
            feature_index
        )

    predictions = collections.OrderedDict()

    for example_index, example in enumerate(tqdm(examples)):
        feature_indices = features_per_example[example_index]
        context = example["context"]
        valid_answers = []
        min_null_score = None

        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]

            offset_mapping = features[feature_index][
                "offset_mapping"
            ]

            # 无答案分数使用 [CLS] 的起止分数
            cls_index = features[feature_index][
                "input_ids"
            ].index(tokenizer.cls_token_id)

            null_score = (
                start_logits[cls_index]
                + end_logits[cls_index]
            )

            if (
                min_null_score is None
                or null_score < min_null_score
            ):
                min_null_score = null_score

            start_indexes = np.argsort(
                start_logits
            )[-1 : -n_best_size - 1 : -1].tolist()

            end_indexes = np.argsort(
                end_logits
            )[-1 : -n_best_size - 1 : -1].tolist()

            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                    ):
                        continue

                    if (
                        end_index < start_index
                        or end_index - start_index + 1
                        > max_answer_length
                    ):
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]

                    valid_answers.append(
                        {
                            "score": (
                                start_logits[start_index]
                                + end_logits[end_index]
                            ),
                            "text": context[
                                start_char:end_char
                            ],
                        }
                    )

        if valid_answers:
            best_answer = max(
                valid_answers,
                key=lambda answer: answer["score"],
            )
        else:
            best_answer = {"text": "", "score": 0.0}

        # 最佳有答案分数没有超过无答案分数
        if best_answer["score"] <= min_null_score:
            predictions[example["id"]] = ""
        else:
            predictions[example["id"]] = best_answer["text"]

    return predictions

## 预测并计算 SQuAD v2 指标

In [12]:
raw_predictions = trainer.predict(
    validation_features
)

validation_features.set_format(
    type=validation_features.format["type"],
    columns=list(validation_features.features.keys()),
)

final_predictions = postprocess_qa_predictions(
    datasets["validation"],
    validation_features,
    raw_predictions.predictions,
)

  0%|          | 0/11873 [00:00<?, ?it/s]

In [13]:
import evaluate

squad_v2_metric = evaluate.load("squad_v2")

formatted_predictions = [
    {
        "id": example_id,
        "prediction_text": prediction_text,
        "no_answer_probability": 0.0,
    }
    for example_id, prediction_text
    in final_predictions.items()
]

references = [
    {
        "id": example["id"],
        "answers": example["answers"],
    }
    for example in datasets["validation"]
]

result = squad_v2_metric.compute(
    predictions=formatted_predictions,
    references=references,
)

print(result)

{'exact': 63.63176956118925, 'f1': 67.1272332084519, 'total': 11873, 'HasAns_exact': 66.58232118758434, 'HasAns_f1': 73.58327258501139, 'HasAns_total': 5928, 'NoAns_exact': 60.689655172413794, 'NoAns_f1': 60.689655172413794, 'NoAns_total': 5945, 'best_exact': 63.64019203234229, 'best_exact_thresh': 0.0, 'best_f1': 67.13565567960514, 'best_f1_thresh': 0.0}


### 参数含义

{'exact': 63.63176956118925, 'f1': 67.1272332084519, 'total': 11873, 'HasAns_exact': 66.58232118758434, 'HasAns_f1': 73.58327258501139, 'HasAns_total': 5928, 'NoAns_exact': 60.689655172413794, 'NoAns_f1': 60.689655172413794, 'NoAns_total': 5945, 'best_exact': 63.64019203234229, 'best_exact_thresh': 0.0, 'best_f1': 67.13565567960514, 'best_f1_thresh': 0.0}

这些指标，是 SQuAD v2 官方评估脚本的标准输出，它们从不同维度刻画了模型的问答能力。

| 指标 | 结果 | 含义 |
| :--- | :--- | :--- |
| **`exact`** | **63.63%** | **精确匹配率（EM）**。模型预测的答案和标准答案**完全一样**（经过标准化处理后）的比例。这是最严格的指标。 |
| **`f1`** | **67.13%** | **F1分数**。模型预测答案和标准答案在**词级别上的重叠程度**，可以理解为“部分正确”的得分。它比EM更宽松。 |
| **`total`** | 11873 | 验证集中**总问题数**。这是计算所有指标的基数。 |
| **`HasAns_exact`** | 66.58% | **可回答问题**的精确匹配率。衡量模型在**确实有答案**的问题上，找得有多准。 |
| **`HasAns_f1`** | 73.58% | **可回答问题**的F1分数。衡量模型在有答案的问题上，答案的**词级重叠度**。 |
| **`HasAns_total`** | 5928 | 验证集中**有答案**的问题总数。约占一半（5928/11873）。 |
| **`NoAns_exact`** | 60.69% | **不可回答问题**的精确匹配率。衡量模型**正确判断出“问题无答案”**的能力。 |
| **`NoAns_f1`** | 60.69% | **不可回答问题**的F1分数。对于“无答案”问题，F1通常与EM相同（判断正确得1分，错误得0分）。 |
| **`NoAns_total`** | 5945 | 验证集中**无答案**的问题总数。与有答案问题数量基本持平。 |
| **`best_exact`** | 63.64% | **最佳精确匹配率**。通过**调整“无答案”的判定阈值**，所能达到的最高EM分数。 |
| **`best_exact_thresh`** | 0.0 | 达到 `best_exact` 时对应的**无答案概率阈值**。阈值为0.0，意味着模型倾向于**极不情愿**地判断问题无答案。 |
| **`best_f1`** | 67.14% | **最佳F1分数**。通过调整阈值能达到的最高F1分数。 |
| **`best_f1_thresh`** | 0.0 | 达到 `best_f1` 时对应的**无答案概率阈值**。 |

### 训练效果如何？

结论：模型表现处于“及格”水平，但仍有很大提升空间。

与官方基线对比：SQuAD v2 原论文中，一个强基线模型的成绩是 F1 66.3%, EM 63.4%。你的模型（F1 67.13%, EM 63.63%）刚好超过了这个基线，说明模型确实学到了东西，不是一个“随机猜测”的模型。

与人类表现对比：SQuAD v2 数据集中，人类的表现大约是 F1 89.5%, EM 86.8%。你的模型与人类水平还有超过20个百分点的巨大差距，说明在深层语义理解和推理方面，模型还有很长的路要走。

与当前先进模型对比：作为参考，在SQuAD v2上，经过良好微调的 BERT-base 模型通常能达到 EM 71%+, F1 74%+ 的水平。你的模型（EM 63.63%, F1 67.13%）与之相比，大概有7-8个百分点的差距，说明训练可能还不够充分，或者超参数配置还有优化空间。

关键问题：判断“无答案”的能力偏弱：

你的模型在可回答问题上的表现（HasAns_f1 73.58%）尚可。

但在不可回答问题上的表现（NoAns_exact 60.69%）明显更差。

best_exact_thresh 和 best_f1_thresh 都为 0.0 是一个关键信号。它意味着模型几乎从不主动预测“无答案”。为了得到最高的分数，评估脚本甚至需要将阈值设为0，暗示模型输出的“无答案”概率普遍非常低。你的模型倾向于强行从文本中截取一个答案，即使这个问题本身没有答案，这说明模型在判断问题可回答性上存在明显不足。

### 提升方向

针对以上分析，你可以尝试以下几个方向来提升模型性能：

- 增加训练轮数 (Epochs)：3个Epoch可能不够。可以尝试训练 5-10个Epoch，同时配合早停（Early Stopping）来防止过拟合。你的 eval_loss 如果还在下降，说明模型还在学习。
- 调整学习率 (Learning Rate)：这是最重要的超参数之一。可以尝试更小的学习率（如 2e-5, 3e-5）进行更精细的微调，或者尝试学习率预热（Warmup）。
- 增大批次大小 (Batch Size)：在显存允许的情况下，适当增大 per_device_train_batch_size（例如从16增加到32），有助于训练更稳定。
- 使用更强的基座模型：你使用的是 distilbert-base-uncased，这是一个蒸馏后的小模型。如果资源允许，换用完整的 bert-base-uncased 或 roberta-base，性能通常会有显著提升（参考上文，BERT-base可达EM 71%+）。
- 深入分析“无答案”问题：这是你模型当前的短板。可以检查验证集中那些被模型错误地回答了的“无答案”问题，看看模型是否容易被相似的干扰项迷惑，并思考如何通过改进数据或训练策略来增强模型的判别能力。

# 原模型继续训练

In [15]:
import glob
import os
import torch

from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)

base_model_path = (
    "/sigan/ai-ml/models/finetuned/"
    "distilbert-base-uncased-finetuned-squad/QA"
)

continued_dir = (
    "/sigan/ai-ml/models/finetuned/"
    "distilbert-base-uncased-finetuned-squad-continued"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForQuestionAnswering.from_pretrained(
    base_model_path,
    local_files_only=True,
).to(device)

try:
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_path,
        local_files_only=True,
    )
except OSError:
    tokenizer = AutoTokenizer.from_pretrained(
        "distilbert-base-uncased"
    )

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at /sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad/QA and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
continued_args = TrainingArguments(
    output_dir=continued_dir,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    report_to="none",
)

continued_trainer = Trainer(
    model=model,
    args=continued_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)

In [17]:
continued_trainer.train()

Epoch,Training Loss,Validation Loss
1,1.045800,1.251804
2,0.933800,1.263095


TrainOutput(global_step=16470, training_loss=1.0493926802764317, metrics={'train_runtime': 1946.5692, 'train_samples_per_second': 135.37, 'train_steps_per_second': 8.461, 'total_flos': 2.582110281242419e+16, 'train_loss': 1.0493926802764317, 'epoch': 2.0})

In [18]:
# 训练结束后，load_best_model_at_end=True 会让内存中保留验证损失最低的 checkpoint
continued_trainer.save_model(
    f"{continued_dir}/best"
)
tokenizer.save_pretrained(
    f"{continued_dir}/best"
)

('/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad-continued/best/tokenizer_config.json',
 '/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad-continued/best/special_tokens_map.json',
 '/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad-continued/best/vocab.txt',
 '/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad-continued/best/added_tokens.json',
 '/sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad-continued/best/tokenizer.json')

## 随后用 v2 的同一套流程评估

In [19]:
raw_predictions = continued_trainer.predict(
    validation_features
)

validation_features.set_format(
    type=validation_features.format["type"],
    columns=list(validation_features.features.keys()),
)

continued_predictions = postprocess_qa_predictions(
    datasets["validation"],
    validation_features,
    raw_predictions.predictions,
)

formatted_predictions = [
    {
        "id": example_id,
        "prediction_text": prediction_text,
        "no_answer_probability": 0.0,
    }
    for example_id, prediction_text
    in continued_predictions.items()
]

continued_result = squad_v2_metric.compute(
    predictions=formatted_predictions,
    references=references,
)

print(continued_result)

  0%|          | 0/11873 [00:00<?, ?it/s]

{'exact': 61.28190010949213, 'f1': 64.87147540657871, 'total': 11873, 'HasAns_exact': 65.62078272604589, 'HasAns_f1': 72.81022731145528, 'HasAns_total': 5928, 'NoAns_exact': 56.95542472666106, 'NoAns_f1': 56.95542472666106, 'NoAns_total': 5945, 'best_exact': 61.29032258064516, 'best_exact_thresh': 0.0, 'best_f1': 64.87989787773198, 'best_f1_thresh': 0.0}


## 判断新模型是否值得保留

如果 F1 和 EM 都提高，通常可以保留。如果 HasAns_f1 提高、NoAns_f1 明显下降，说明模型更会找答案，但变得不愿意回答“不知道”，需要重新考虑无答案阈值。

In [21]:
if (
    continued_result["f1"] > result["f1"]
    and continued_result["HasAns_f1"]
    > result["HasAns_f1"]
    and continued_result["NoAns_f1"]
    >= result["NoAns_f1"]
):
    print("两个能力都提升，可以保留新模型")
else:
    print("至少一个关键指标没有提升，需要继续比较")

至少一个关键指标没有提升，需要继续比较


# 加载模型并提问

In [22]:
import torch

from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
)

model_path = (
    "/sigan/ai-ml/models/finetuned/"
    "distilbert-base-uncased-finetuned-squad-v2"
)

# 加载的是刚训练的 v2 模型
SQUAD_V2_MODEL = True

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForQuestionAnswering.from_pretrained(
    model_path,
    local_files_only=True,
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True,
)

model.eval()

DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
      

In [23]:
# 单次回答函数
def ask_with_context(
    question,
    context,
    n_best_size=20,
    max_answer_length=30,
):
    encoded = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=max_length,
        return_offsets_mapping=True,
    )

    offset_mapping = encoded.pop(
        "offset_mapping"
    )[0].tolist()

    sequence_ids = encoded.sequence_ids(0)

    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.no_grad():
        outputs = model(**encoded)

    start_logits = outputs.start_logits[0].cpu()
    end_logits = outputs.end_logits[0].cpu()

    start_indexes = torch.topk(
        start_logits,
        n_best_size,
    ).indices.tolist()

    end_indexes = torch.topk(
        end_logits,
        n_best_size,
    ).indices.tolist()

    candidates = []

    for start_index in start_indexes:
        for end_index in end_indexes:
            # 答案必须来自 context
            if sequence_ids[start_index] != 1:
                continue
            if sequence_ids[end_index] != 1:
                continue

            if end_index < start_index:
                continue
            if (
                end_index - start_index + 1
                > max_answer_length
            ):
                continue

            score = (
                start_logits[start_index]
                + end_logits[end_index]
            ).item()

            start_char = offset_mapping[start_index][0]
            end_char = offset_mapping[end_index][1]

            candidates.append(
                {
                    "score": score,
                    "text": context[start_char:end_char],
                }
            )

    if not candidates:
        return ""

    best_answer = max(
        candidates,
        key=lambda answer: answer["score"],
    )

    if SQUAD_V2_MODEL:
        input_ids = tokenizer.convert_tokens_to_ids(
            tokenizer.cls_token
        )

        cls_index = encoded["input_ids"][0].tolist().index(
            input_ids
        )

        null_score = (
            start_logits[cls_index]
            + end_logits[cls_index]
        ).item()

        if best_answer["score"] <= null_score:
            return ""

    return best_answer["text"]

In [24]:
# 有答案的测试题
test_cases = [
    {
        "question": (
            "Which team was eliminated "
            "from the tournament?"
        ),
        "context": (
            "The Shanghai Dragons lost to Beijing "
            "in the semifinal and were eliminated "
            "from the tournament."
        ),
        "expected": "The Shanghai Dragons",
    },
    {
        "question": "Where was the final match held?",
        "context": (
            "The final match was held in Shanghai "
            "on October 26."
        ),
        "expected": "Shanghai",
    },
    {
        "question": "Who scored the winning goal?",
        "context": (
            "Li Ming scored the winning goal "
            "in the final minute."
        ),
        "expected": "Li Ming",
    },
    {
        "question": "When was the match held?",
        "context": (
            "The final match was held in Shanghai "
            "on October 26."
        ),
        "expected": "October 26",
    },
]

In [25]:
for i, case in enumerate(test_cases, start=1):
    answer = ask_with_context(
        case["question"],
        case["context"],
    )

    print(f"测试 {i}")
    print("问题：", case["question"])
    print("回答：", answer)
    print("答案原文：", case["expected"])
    print("-" * 40)

测试 1
问题： Which team was eliminated from the tournament?
回答： The Shanghai Dragons
答案原文： The Shanghai Dragons
----------------------------------------
测试 2
问题： Where was the final match held?
回答： Shanghai
答案原文： Shanghai
----------------------------------------
测试 3
问题： Who scored the winning goal?
回答： Li Ming
答案原文： Li Ming
----------------------------------------
测试 4
问题： When was the match held?
回答： October 26
答案原文： October 26
----------------------------------------


In [26]:
# 无答案的测试题
answer = ask_with_context(
    question="How many tickets were sold?",
    context=(
        "The final match was held in Shanghai "
        "on October 26. Li Ming scored the winning goal."
    ),
)

print(repr(answer))

''


# squad-v2继续训练

In [27]:
import collections
import glob
import math
import os
import re
from pathlib import Path

import evaluate
import numpy as np
import torch
from datasets import load_dataset
from tqdm.auto import tqdm

from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)


# 修改为你的 v2 模型目录
base_model_path = Path(
    "/sigan/ai-ml/models/finetuned/"
    "distilbert-base-uncased-finetuned-squad-v2"
)

# 第二阶段的输出目录
continued_dir = Path(
    "/sigan/ai-ml/models/finetuned/"
    "distilbert-base-uncased-finetuned-squad-v2-stage2"
)

continued_dir.mkdir(parents=True, exist_ok=True)


model = AutoModelForQuestionAnswering.from_pretrained(
    base_model_path,
    local_files_only=True,
)


try:
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_path,
        local_files_only=True,
    )
except OSError:
    # 问答微调不会改变 tokenizer 规则
    tokenizer = AutoTokenizer.from_pretrained(
        "distilbert-base-uncased"
    )


device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("设备：", device)
print("模型：", model.__class__.__name__)

设备： cuda
模型： DistilBertForQuestionAnswering


In [28]:
squad_v2 = True
max_length = 384
doc_stride = 128
pad_on_right = tokenizer.padding_side == "right"

datasets = load_dataset("squad_v2")

print(datasets)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})


In [29]:
def prepare_train_features(examples):
    examples["question"] = [
        question.lstrip()
        for question in examples["question"]
    ]

    tokenized_examples = tokenizer(
        examples["question" if pad_on_right else "context"],
        examples["context" if pad_on_right else "question"],
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop(
        "overflow_to_sample_mapping"
    )
    offset_mapping = tokenized_examples.pop(
        "offset_mapping"
    )

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        sequence_ids = tokenized_examples.sequence_ids(i)

        cls_index = input_ids.index(
            tokenizer.cls_token_id
        )

        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        # SQuAD v2 的无答案样本
        if len(answers["answer_start"]) == 0:
            tokenized_examples[
                "start_positions"
            ].append(cls_index)
            tokenized_examples[
                "end_positions"
            ].append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = (
            start_char + len(answers["text"][0])
        )

        context_sequence_id = 1 if pad_on_right else 0

        token_start_index = 0
        while (
            sequence_ids[token_start_index]
            != context_sequence_id
        ):
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while (
            sequence_ids[token_end_index]
            != context_sequence_id
        ):
            token_end_index -= 1

        # 标准答案不在当前窗口中，也按无答案训练
        answer_in_window = (
            offsets[token_start_index][0] <= start_char
            and offsets[token_end_index][1] >= end_char
        )

        if not answer_in_window:
            tokenized_examples[
                "start_positions"
            ].append(cls_index)
            tokenized_examples[
                "end_positions"
            ].append(cls_index)
            continue

        while (
            token_start_index < len(offsets)
            and offsets[token_start_index][0]
            <= start_char
        ):
            token_start_index += 1

        while (
            offsets[token_end_index][1]
            >= end_char
        ):
            token_end_index -= 1

        tokenized_examples[
            "start_positions"
        ].append(token_start_index - 1)

        tokenized_examples[
            "end_positions"
        ].append(token_end_index + 1)

    return tokenized_examples

In [30]:
tokenized_datasets = datasets.map(
    prepare_train_features,
    batched=True,
    remove_columns=datasets["train"].column_names,
    load_from_cache_file=False,
)

print(tokenized_datasets)

Map:   0%|          | 0/130319 [00:00<?, ? examples/s]

Map:   0%|          | 0/11873 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 131754
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 12134
    })
})


### 继续训练2epoch

In [31]:
batch_size = 16
gradient_accumulation_steps = 4

use_bf16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)

training_args = TrainingArguments(
    output_dir=str(continued_dir),
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    num_train_epochs=2,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    weight_decay=0.01,
    warmup_ratio=0.1,
    bf16=use_bf16,
    fp16=False,
    save_total_limit=3,
    load_best_model_at_end=False,
    logging_steps=100,
    report_to="none",
)

print("BF16：", use_bf16)
print(
    "等效 batch：",
    batch_size * gradient_accumulation_steps
)

BF16： True
等效 batch： 64


In [32]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)

In [33]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss
0,0.561800,1.682027
1,0.564500,1.757084


In [34]:
final_model_dir = continued_dir / "final"
final_model_dir.mkdir(
    parents=True,
    exist_ok=True,
)

trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))

print("模型已保存：", final_model_dir)

模型已保存： /sigan/ai-ml/models/finetuned/distilbert-base-uncased-finetuned-squad-v2-stage2/final


In [35]:
def prepare_validation_features(examples):
    examples["question"] = [
        question.lstrip()
        for question in examples["question"]
    ]

    tokenized_examples = tokenizer(
        examples["question" if pad_on_right else "context"],
        examples["context" if pad_on_right else "question"],
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop(
        "overflow_to_sample_mapping"
    )

    tokenized_examples["example_id"] = []

    for i in range(
        len(tokenized_examples["input_ids"])
    ):
        sequence_ids = tokenized_examples.sequence_ids(i)
        context_index = 1 if pad_on_right else 0
        sample_index = sample_mapping[i]

        tokenized_examples["example_id"].append(
            examples["id"][sample_index]
        )

        tokenized_examples["offset_mapping"][i] = [
            offset
            if sequence_ids[k] == context_index
            else None
            for k, offset in enumerate(
                tokenized_examples["offset_mapping"][i]
            )
        ]

    return tokenized_examples

In [36]:
validation_features = datasets["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=datasets[
        "validation"
    ].column_names,
    load_from_cache_file=False,
)

Map:   0%|          | 0/11873 [00:00<?, ? examples/s]

### 输出答案概率

In [37]:
def sigmoid(value):
    if value >= 0:
        z = math.exp(-value)
        return 1.0 / (1.0 + z)

    z = math.exp(value)
    return z / (1.0 + z)

In [38]:
def postprocess_with_null_probability(
    examples,
    features,
    raw_predictions,
    n_best_size=20,
    max_answer_length=30,
):
    all_start_logits, all_end_logits = raw_predictions

    example_id_to_index = {
        example_id: index
        for index, example_id in enumerate(examples["id"])
    }

    features_per_example = collections.defaultdict(list)

    for feature_index, feature in enumerate(features):
        original_index = example_id_to_index[
            feature["example_id"]
        ]
        features_per_example[original_index].append(
            feature_index
        )

    predictions = collections.OrderedDict()

    for example_index, example in enumerate(
        tqdm(examples)
    ):
        context = example["context"]
        feature_indices = features_per_example[
            example_index
        ]

        min_null_score = None
        valid_answers = []

        for feature_index in feature_indices:
            start_logits = all_start_logits[
                feature_index
            ]
            end_logits = all_end_logits[
                feature_index
            ]

            offset_mapping = features[
                feature_index
            ]["offset_mapping"]

            input_ids = features[
                feature_index
            ]["input_ids"]

            cls_index = input_ids.index(
                tokenizer.cls_token_id
            )

            null_score = float(
                start_logits[cls_index]
                + end_logits[cls_index]
            )

            if (
                min_null_score is None
                or null_score < min_null_score
            ):
                min_null_score = null_score

            start_indexes = np.argsort(
                start_logits
            )[-1 : -n_best_size - 1 : -1].tolist()

            end_indexes = np.argsort(
                end_logits
            )[-1 : -n_best_size - 1 : -1].tolist()

            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index]
                        is None
                        or offset_mapping[end_index]
                        is None
                    ):
                        continue

                    if end_index < start_index:
                        continue

                    if (
                        end_index - start_index + 1
                        > max_answer_length
                    ):
                        continue

                    score = float(
                        start_logits[start_index]
                        + end_logits[end_index]
                    )

                    start_char = offset_mapping[
                        start_index
                    ][0]
                    end_char = offset_mapping[
                        end_index
                    ][1]

                    valid_answers.append(
                        {
                            "score": score,
                            "text": context[
                                start_char:end_char
                            ],
                        }
                    )

        if valid_answers:
            best_answer = max(
                valid_answers,
                key=lambda answer: answer["score"],
            )
            answer_score = best_answer["score"]
            answer_text = best_answer["text"]

            null_probability = sigmoid(
                min_null_score - answer_score
            )
        else:
            answer_text = ""
            null_probability = 1.0

        predictions[example["id"]] = {
            "answer": answer_text,
            "null_probability": float(
                null_probability
            ),
        }

    return predictions

In [39]:
raw_predictions = trainer.predict(
    validation_features
)

validation_features.set_format(
    type="python",
    columns=list(
        validation_features.features.keys()
    ),
)

validation_predictions = (
    postprocess_with_null_probability(
        datasets["validation"],
        validation_features,
        raw_predictions.predictions,
    )
)

  0%|          | 0/11873 [00:00<?, ?it/s]

In [40]:
validation_split = datasets[
    "validation"
].train_test_split(
    test_size=0.5,
    seed=42,
)

tune_examples = validation_split["train"]
report_examples = validation_split["test"]

print("调阈值样本数：", len(tune_examples))
print("最终报告样本数：", len(report_examples))

调阈值样本数： 5936
最终报告样本数： 5937


In [41]:
references_by_id = {
    example["id"]: {
        "id": example["id"],
        "answers": example["answers"],
    }
    for example in datasets["validation"]
}

squad_v2_metric = evaluate.load("squad_v2")


def make_metric_predictions(
    example_ids,
    threshold,
):
    formatted = []

    for example_id in example_ids:
        prediction = validation_predictions[
            example_id
        ]

        if (
            prediction["null_probability"]
            >= threshold
        ):
            prediction_text = ""
        else:
            prediction_text = prediction["answer"]

        formatted.append(
            {
                "id": example_id,
                "prediction_text": prediction_text,
                "no_answer_probability": (
                    prediction["null_probability"]
                ),
            }
        )

    return formatted


def make_references(example_ids):
    return [
        references_by_id[example_id]
        for example_id in example_ids
    ]


tune_ids = tune_examples["id"]
report_ids = report_examples["id"]

tune_references = make_references(tune_ids)
report_references = make_references(report_ids)

# 选择阈值
threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.02):
    tune_predictions = make_metric_predictions(
        tune_ids,
        threshold,
    )

    result = squad_v2_metric.compute(
        predictions=tune_predictions,
        references=tune_references,
    )

    threshold_results.append(
        {
            "threshold": float(threshold),
            **result,
        }
    )


best_threshold_result = max(
    threshold_results,
    key=lambda result: result["f1"],
)

best_threshold = best_threshold_result[
    "threshold"
]

print("最佳阈值：", best_threshold)
print(best_threshold_result)

最佳阈值： 0.22
{'threshold': 0.22, 'exact': 64.52156334231806, 'f1': 68.10607672250983, 'total': 5936, 'HasAns_exact': 65.04010695187166, 'HasAns_f1': 72.1516281500064, 'HasAns_total': 2992, 'NoAns_exact': 63.994565217391305, 'NoAns_f1': 63.994565217391305, 'NoAns_total': 2944, 'best_exact': 65.98719676549865, 'best_exact_thresh': 0.012431650422513485, 'best_f1': 68.99052548814683, 'best_f1_thresh': 0.031143831089138985}


In [42]:
report_predictions = make_metric_predictions(
    report_ids,
    best_threshold,
)

final_result = squad_v2_metric.compute(
    predictions=report_predictions,
    references=report_references,
)

print(final_result)

{'exact': 64.32541687721071, 'f1': 67.62462612413978, 'total': 5937, 'HasAns_exact': 65.25885558583106, 'HasAns_f1': 71.93031515634141, 'HasAns_total': 2936, 'NoAns_exact': 63.412195934688434, 'NoAns_f1': 63.412195934688434, 'NoAns_total': 3001, 'best_exact': 65.43708943911066, 'best_exact_thresh': 0.01590639166533947, 'best_f1': 68.3039251501572, 'best_f1_thresh': 0.03622005507349968}


### 结论

1. v2 再训练版目前略占优势，但胜在无答案能力；v2 原版更擅长有答案问答。还不能仅凭这组不同样本量的结果决定最终保留哪一版。
2. 现阶段不建议继续增加 epoch。
   - 再训练 2 epoch 后：
     - HasAns F1 下降了 1.65。
     - Overall F1 只提高 0.50。
     - 提升主要来自 NoAns 判断，而不是答案抽取能力本身。
     - 继续训练很可能进一步偏向“少回答”，让 HasAns 继续下降。
3. 目前更值得做的是公平复测，而不是继续训练：
   1. 用同一个 5,937 条 report 验证集。
   2. 用同一套 no_answer_probability 计算代码。
   3. 对原版和再训练版使用同一个固定阈值，例如：threshold = 0.03622005507349968
   4. 比较两个模型在同一批问题上的 Exact、F1、HasAns_F1、NoAns_F1。
   5. 优先看成对差异，而不是只看两个总分。
4. 如果公平复测后结果仍是：
   - 再训练版 Overall F1 更高
   - 再训练版 NoAns_F1 更高
   - HasAns_F1 只下降约 1 个点

##### 那么可以保留再训练版。如果公平复测后 Overall F1 差距消失，或者 HasAns 下降更明显，就保留 v2 原版。